## Cell 1 — Load Dataset

In [ ]:
from datasets import Dataset
import json, os

DATASET_DIR = '/kaggle/input/datasets/mrafay7/islamic-ai-dataset'

def load_jsonl(path):
    pairs = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                pairs.append(json.loads(line))
    return pairs

print('Loading JSONL files...')
train_pairs = load_jsonl(f'{DATASET_DIR}/train.jsonl')
eval_pairs  = load_jsonl(f'{DATASET_DIR}/eval.jsonl')
print(f'Train: {len(train_pairs):,} pairs')
print(f'Eval : {len(eval_pairs):,} pairs')
print('✅ Loaded.')

## Cell 2 — System Prompt

In [ ]:
SYSTEM_PROMPT = """You are Noor (نور), an Islamic AI assistant. Your name means 'Light', inspired by: "Allah is the Light of the heavens and the earth." (Quran An-Nur 24:35)

You are a learning tool — NOT a qualified mufti or Islamic scholar. Always clarify this for complex personal matters.

RESPONSE STRUCTURE — Always answer in this exact order with FULL DETAIL in every section:

**Explanation**
Write 2–4 paragraphs. Cover:
- What this topic is in Islamic terminology (define the Arabic term and its meaning)
- Why it matters in Islam and how it fits into the broader deen
- Historical or contextual background where relevant
- Any important conditions, types, or categories that affect the ruling

**Quranic Evidence**
For each relevant ayah:
- Arabic text in Arabic script (e.g. ﴿وَٱلْخَيْلَ وَٱلْبِغَالَ﴾)
- Full English translation
- — Quran, [Surah Name] ([Surah Number]:[Ayah Number])
- Brief tafsir and Asbab al-Nuzul if known
Cite at least 2–3 ayahs where they exist.

**Hadith Evidence**
For each relevant hadith:
- Full hadith text with narrator name and RA/ﷺ
- — [Book Name], Hadith [Number] ([Grade: Sahih / Hasan / Da'if])
- One sentence on what this hadith proves
Cite at least 2–3 hadith. Mark Da'if with ⚠️

**Scholarly Positions (Fiqh)**
For EACH of the four madhabs:
• Hanafi: ruling + dalil + conditions
• Maliki: ruling + dalil + conditions
• Shafi'i: ruling + dalil + conditions
• Hanbali: ruling + dalil + conditions
Mark [IJMA] if all agree. Mark [KHILAF] if they differ.

**Summary**
2–3 paragraphs synthesizing positions + practical guidance.
End with: "For your specific situation, please consult a qualified Islamic scholar (mufti or imam)."

CITATIONS — MANDATORY:
- Write ﷺ after Prophet Muhammad
- Write RA after companions
- Write RH after classical scholars
- Never cite hadith without book name and number
- Never cite Quran without Surah name, number, and ayah number

SAFETY:
- NEVER issue binding fatwas
- NEVER declare anyone kafir
- NEVER fabricate Islamic content
- For haram requests: refuse + explain + offer halal alternative

UNCERTAINTY:
- Say "Allahu Akbar, and Allah knows best" when uncertain"""

print(f'System prompt ready. {len(SYSTEM_PROMPT)} chars')

## Cell 3 — Install Packages

In [ ]:
%%capture
# --no-deps only for unsloth/unsloth_zoo to avoid upgrading transformers/huggingface_hub
# trl, peft, accelerate, bitsandbytes install normally (they don't cause version conflicts)
!pip install -q --no-deps unsloth_zoo unsloth
!pip install -q trl peft accelerate bitsandbytes


In [ ]:
# Patch missing symbols in huggingface_hub (Kaggle version compatibility)
import huggingface_hub.hf_api as _hf
if not hasattr(_hf, 'KernelInfo'):
    class KernelInfo: pass
    _hf.KernelInfo = KernelInfo
    print('KernelInfo patched.')
else:
    print('KernelInfo OK.')

import huggingface_hub.utils.tqdm as _tqdm
if not hasattr(_tqdm, '_create_progress_bar'):
    from tqdm.auto import tqdm as _tqdm_cls
    _tqdm._create_progress_bar = lambda *a, **kw: _tqdm_cls(*a, **kw)
    print('_create_progress_bar patched.')
else:
    print('_create_progress_bar OK.')


## Cell 4 — Load Model + Format Dataset

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset

MAX_SEQ_LENGTH = 2048

_, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

def format_pair(pair):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": pair["instruction"].strip()},
        {"role": "assistant", "content": pair["output"].strip()},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

def strip_pair(p):
    return {"instruction": p["instruction"], "output": p["output"]}

print('Formatting dataset...')
train_dataset = Dataset.from_list([strip_pair(p) for p in train_pairs]).map(format_pair, desc='Train')
eval_dataset  = Dataset.from_list([strip_pair(p) for p in eval_pairs]).map(format_pair,  desc='Eval')
print(f'Train: {len(train_dataset):,} | Eval: {len(eval_dataset):,}')
print('✅ Done')

## Cell 5 — Load Full Model + Apply LoRA

In [ ]:
import torch

print('Loading Llama 3.2 3B...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha                 = 16,
    lora_dropout               = 0,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state               = 42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M ({100*trainable/total:.2f}%)')
print('✅ Model + LoRA ready.')

## Cell 6 — Configure Trainer

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

OUTPUT_DIR = '/kaggle/working/outputs'
SAVE_DIR   = '/kaggle/working/model/lora'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,   exist_ok=True)

trainer = SFTTrainer(
    model              = model,
    train_dataset      = train_dataset,
    dataset_text_field = 'text',
    args = SFTConfig(
        max_seq_length              = MAX_SEQ_LENGTH,
        dataset_num_proc            = 2,
        packing                     = True,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        num_train_epochs            = 2,
        warmup_steps                = 100,
        learning_rate               = 2e-4,
        lr_scheduler_type           = 'cosine',
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        weight_decay                = 0.01,
        optim                       = 'adamw_8bit',
        logging_steps               = 25,
        report_to                   = 'none',
        eval_strategy               = 'no',
        save_strategy               = 'steps',
        save_steps                  = 500,
        save_total_limit            = 1,
        output_dir                  = OUTPUT_DIR,
        seed                        = 42,
    ),
)

steps_per_epoch = len(train_dataset) // (2 * 8)
total_steps     = steps_per_epoch * 2
print(f'Steps per epoch : {steps_per_epoch}')
print(f'Total steps     : {total_steps}')
print(f'Estimated time  : {total_steps * 6 / 3600:.1f} hours on T4')
print('Trainer ready.')


## Cell 7 — Start Training (8–10 hours)

In [ ]:
print('Starting training... this will take 8-10 hours.')
train_result = trainer.train()
print('✅ Training complete!')
print(f'Loss: {train_result.training_loss:.4f}')

## Cell 8 — Save Model

In [ ]:
print(f'Saving model to {SAVE_DIR}...')
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print('✅ Model saved!')
print('Files:')
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1e6
    print(f'  {f} ({size:.1f} MB)')